In [ ]:
import os

import geopandas as gpd
import pandas as pd

from xyzservices import TileProvider

from zwerfafval_detectie.utils_eval import read_annotations_folder


RD_CRS = "EPSG:28992"  # CRS code for the Dutch Rijksdriehoek coordinate system
LAT_LON_CRS = "EPSG:4326"  # CRS code for WGS84 latitude/longitude coordinate system

ams_tile_provider = TileProvider(
    name="Topografie, standaard visualisatie (WM)",
    url="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attribution="data.amsterdam.nl",
)

In [ ]:
model = "yolo26m_1920_v1-2_extra_250"
split = "train"

predictions_folder = f"../datasets/experiments/zwerfafval/predict/{model}/{split}"

categories = {
    0: "Zwerfafval (grof)",
    1: "Zwerfafval (fijn)"
}

confidence = 0.2

In [ ]:
predictions_gdf = read_annotations_folder(folder_path=predictions_folder, categories=categories)
predictions_gdf["file_name"] = predictions_gdf["file_name"].str.replace(".txt", ".jpg")

In [ ]:
_predictions_sorted = (
    predictions_gdf[predictions_gdf["confidence"] >= confidence]
    .set_index("file_name")
    .sort_index()
)

counts_df = (
    _predictions_sorted[["category"]]
    .replace(categories)
    .groupby(["file_name", "category"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
metadata_files = [
    "../datasets/experiments/zwerfafval/annotatieproject/inwinning_250514_selectie_300.gpkg",
    "../datasets/experiments/zwerfafval/annotatieproject/inwinning_260421_selectie_1000.gpkg"
]

metadata_gdf = pd.concat([
    gpd.read_file(metadata_file, layer=0)
    for metadata_file in metadata_files
]).set_index("file_name")

In [ ]:
counts_merged = gpd.GeoDataFrame(counts_df.join(metadata_gdf, how="left"))

In [ ]:
counts_merged = counts_merged[["Zwerfafval (fijn)", "Zwerfafval (grof)", "geometry"]].to_crs(RD_CRS).dropna()

In [ ]:
map = (
    counts_merged
    .explore(
        column="Zwerfafval (grof)",
        cmap="YlOrRd",
        style_kwds={
            "style_function": lambda x: {"radius": 2*x["properties"]["Zwerfafval (grof)"]},
            "fillOpacity": 0.75,
            "weight": 2
        },
        legend=True,
        tiles=ams_tile_provider
    )
)

map.save(os.path.join("../datasets/experiments/zwerfafval", f"heatmap_v1_{model}_{split}.html"))

In [ ]:
counts_merged

In [ ]:
import numpy as np
import shapely.geometry as sg

# total area for the grid
xmin, ymin, xmax, ymax = counts_merged.total_bounds

xmin = int(xmin / 100) * 100
ymin = int(ymin / 100) * 100
xmax = int((xmax + 100) / 100) * 100
ymax = int((ymax + 100) / 100) * 100

# how many cells across and down
cell_size = 100

# create the cells in a loop
grid_cells = []
for x0 in np.arange(xmin, xmax+cell_size, cell_size ):
    for y0 in np.arange(ymin, ymax+cell_size, cell_size):
        # bounds
        x1 = x0-cell_size
        y1 = y0+cell_size
        grid_cells.append( sg.box(x0, y0, x1, y1)  )
grid = gpd.GeoDataFrame(grid_cells, columns=['geometry'], crs=RD_CRS)

In [ ]:
merged = gpd.sjoin(counts_merged, grid, how='left', predicate='within')

In [ ]:
dissolve = merged.dissolve(by="index_right", aggfunc="sum")

In [ ]:
grid.loc[dissolve.index, "Zwerfafval (grof)"] = dissolve["Zwerfafval (grof)"].values

In [ ]:
grid = grid.dropna()

In [ ]:
grid.plot(column="Zwerfafval (grof)", figsize=(12, 8), cmap='YlOrRd', edgecolor="grey")

In [ ]:
map = (
    grid
    .explore(
        column="Zwerfafval (grof)",
        cmap="YlOrRd",
        style_kwds={
            "fillOpacity": 0.75,
            "weight": 2
        },
        legend=True,
        tiles=ams_tile_provider
    )
)

map.save(os.path.join("../datasets/experiments/zwerfafval", f"heatmap_v1_{model}_{split}_grid.html"))